# 22. Exercise Definition Test (Pipeline 03)

- Goal: check selected exercise-definition loading, split YAML resolution, and pipeline Step 3 integration.
- Docs: docs_eng/pipeline/03_exercise_definition.md / docs/pipeline/03_exercise_definition.md
- Inputs: canonical or authoring-draft exercise definition YAML files, plus p01 pose/annotation CSV for the integration check.
- Outputs: selected definition summary and exercise_definition report from run_pipeline().
- Verification points: selected definition is not generic fallback; key fields are populated; camera protocol matches recording metadata when available; pipeline report contains exercise-definition provenance.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

import warnings

from movement.exercise_definition import load_exercise_definition

DEFINITIONS_DIR = PROJECT_ROOT / "data/definitions/exercises"
AUTHORING_EXAMPLE_ROOT = PROJECT_ROOT / "data/examples/exercise_authoring"
AUTHORING_DRAFT_ROOT = PROJECT_ROOT / "data/processed/authoring_drafts"

# Default stage-check target. Change to "draft_squat" to test an authoring draft.
TARGET_EXERCISE_ID = "draft_squat"


def candidate_definition_dirs(exercise_id):
    return [
        DEFINITIONS_DIR,
        AUTHORING_DRAFT_ROOT / exercise_id / "data/definitions/exercises",
        AUTHORING_EXAMPLE_ROOT / exercise_id / "data/definitions/exercises",
    ]


def resolve_target_definitions_dir(exercise_id):
    for definitions_dir in candidate_definition_dirs(exercise_id):
        if (definitions_dir / f"{exercise_id}.yaml").exists():
            return definitions_dir
    return DEFINITIONS_DIR


TARGET_DEFINITIONS_DIR = resolve_target_definitions_dir(TARGET_EXERCISE_ID)


## Case 1: Load Selected Definition

For the current p01 stage-check path, load one selected definition first.
The default target is `squat` because the sample annotation uses `exercise_type: squat`.
If `TARGET_EXERCISE_ID` is changed to an authoring draft such as `draft_squat`,
the notebook resolves the matching local draft bundle before the git-tracked example bundle.
Broader registry coverage belongs in unit tests or a separate registry audit.

In [3]:
exercise_def = load_exercise_definition(TARGET_EXERCISE_ID, TARGET_DEFINITIONS_DIR)
selected_defs = {exercise_def.exercise_id: exercise_def}

print("loaded selected definition:")
print(f"  exercise_id     : {exercise_def.exercise_id}")
print(f"  version         : {exercise_def.version}")
print(f"  fallback        : {exercise_def.is_generic_fallback}")
print(f"  definitions_dir : {TARGET_DEFINITIONS_DIR}")

loaded selected definition:
  exercise_id     : draft_squat
  version         : draft
  fallback        : False
  definitions_dir : C:\Users\andi9\Desktop\DEV\source\movement_project\data\processed\authoring_drafts\draft_squat\data\definitions\exercises


In [4]:
import pandas as pd

expected_path = TARGET_DEFINITIONS_DIR / f"{TARGET_EXERCISE_ID}.yaml"
assert expected_path.exists(), f"missing selected definition YAML: {expected_path}"
assert exercise_def.exercise_id == TARGET_EXERCISE_ID
assert exercise_def.is_generic_fallback is False

summary = pd.DataFrame([
    {
        'exercise_id': exercise_def.exercise_id,
        'display_name': exercise_def.display_name,
        'definitions_dir': str(TARGET_DEFINITIONS_DIR),
        'is_generic_fallback': exercise_def.is_generic_fallback,
        'laterality': exercise_def.classification.get('laterality'),
        'posture_type': exercise_def.classification.get('posture_type'),
    }
])
display(summary)
print("PASS: selected exercise definition loaded for the current stage-check target")


,exercise_id,display_name,definitions_dir,is_generic_fallback,laterality,posture_type
0,draft_squat,Draft Squat,C:\Users\andi9\Desktop\DEV\source\movement_pro...,False,bilateral_symmetric,standing


PASS: selected exercise definition loaded for the current stage-check target


## Case 2: Selected Definition Field Inspection

Spot-check the most important typed fields for the selected definition.

In [5]:
for ex_id, ed in selected_defs.items():
    clf = ed.classification
    print(f"── {ex_id} ──────────────────────────────")
    print(f"  laterality       : {clf.get('laterality')}")
    print(f"  posture_type     : {clf.get('posture_type')}")
    print(f"  primary_plane    : {clf.get('primary_plane')}")
    print(f"  phase_model.type : {ed.phase_model.type}")
    print(f"  expected_ratio   : {ed.phase_model.expected_ratio}")
    print(f"  primary_joints   : {ed.landmarks.primary_joints}")
    print(f"  compensation_candidates ({len(ed.compensation_candidates)}): {ed.compensation_candidates}")
    print()

── draft_squat ──────────────────────────────
  laterality       : bilateral_symmetric
  posture_type     : standing
  primary_plane    : sagittal
  phase_model.type : resistance_phase
  expected_ratio   : {'eccentric': 0.4, 'isometric': 0.1, 'concentric': 0.5}
  primary_joints   : ['left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle']
  compensation_candidates (9): ['knee_valgus', 'knee_varus', 'asymmetric_depth', 'excessive_trunk_flexion', 'lateral_pelvic_shift', 'heel_lift', 'pelvis_rotation', 'tempo_instability', 'foot_external_rotation_proxy']



## Case 3: Pipeline Step Integration

Run the pipeline through Step 3 with the selected definition and confirm the exercise-definition report.


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.pipeline import load_pipeline_config, run_pipeline

config_path = PROJECT_ROOT / "configs/pipeline_default.yaml"
csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"

config = load_pipeline_config(config_path)
df = load_pose_csv(csv_path)
ann_df = load_annotation_csv(ann_path)

# Step 3 normally receives exercise context from Step 2 annotation when
# exercise_id is not explicitly configured. Enable annotation here so this
# stage check validates that handoff instead of the generic fallback path.
config.annotation.enabled = True
config.annotation.path = ann_path

selected_exercise_id = globals().get("TARGET_EXERCISE_ID", "squat")
selected_definitions_dir = globals().get(
    "TARGET_DEFINITIONS_DIR",
    PROJECT_ROOT / "data/definitions/exercises",
)

# If Case 1 was switched to a draft or another selected definition, run the
# pipeline against that explicit definition. With the default squat target,
# keep the annotation-derived handoff path under test.
if selected_exercise_id != "squat":
    config.exercise_definition.exercise_id = selected_exercise_id
    config.exercise_definition.definitions_dir = str(selected_definitions_dir)

print("annotation.enabled:", config.annotation.enabled)
print("annotation.path:", config.annotation.path)
print("selected_exercise_id:", selected_exercise_id)
print("selected_definitions_dir:", selected_definitions_dir)
print("exercise_definition.enabled:", config.exercise_definition.enabled)
print("exercise_definition.definitions_dir:", config.exercise_definition.definitions_dir)
print("exercise_definition.exercise_id:", config.exercise_definition.exercise_id)

annotation.enabled: True
annotation.path: C:\Users\andi9\Desktop\DEV\source\movement_project\data\pose\mediapipe\no_consent\20260517\p01_squat_set1_annotation.csv
selected_exercise_id: draft_squat
selected_definitions_dir: C:\Users\andi9\Desktop\DEV\source\movement_project\data\processed\authoring_drafts\draft_squat\data\definitions\exercises
exercise_definition.enabled: True
exercise_definition.definitions_dir: C:\Users\andi9\Desktop\DEV\source\movement_project\data\processed\authoring_drafts\draft_squat\data\definitions\exercises
exercise_definition.exercise_id: draft_squat


In [7]:
import warnings as _w

with _w.catch_warnings(record=True) as caught:
    _w.simplefilter("always")
    result_df, report = run_pipeline(df, config=config, landmarks=LANDMARKS, ann_df=ann_df)

print("steps executed:", list(report.keys()))
print()

if caught:
    print(f"{len(caught)} warning(s) during pipeline run:")
    for w in caught:
        print(f"  [{w.category.__name__}] {w.message}")

steps executed: ['validation', 'annotation', 'exercise_definition', 'normalization']



In [8]:
import json

assert "exercise_definition" in report, "exercise_definition step missing from report"
exd_report = report["exercise_definition"]
print(json.dumps(exd_report, indent=2))

annotated_exercise_ids = sorted(
    str(value) for value in ann_df["exercise_type"].dropna().unique()
)
configured_exercise_id = config.exercise_definition.exercise_id
configured_definitions_dir = Path(config.exercise_definition.definitions_dir)
if not configured_definitions_dir.is_absolute():
    configured_definitions_dir = PROJECT_ROOT / configured_definitions_dir
known_definition_ids = {path.stem for path in configured_definitions_dir.glob("*.yaml")}

if configured_exercise_id:
    if configured_exercise_id in known_definition_ids:
        assert exd_report["exercise_id"] == configured_exercise_id
        if configured_exercise_id != "generic":
            assert exd_report["is_generic_fallback"] is False
    else:
        assert exd_report["exercise_id"] == "generic"
        assert exd_report["is_generic_fallback"] is True
else:
    expected_specific_ids = [
        exercise_id
        for exercise_id in annotated_exercise_ids
        if exercise_id in known_definition_ids
    ]
    if expected_specific_ids:
        assert exd_report["exercise_id"] in expected_specific_ids, (
            "exercise_definition did not load the annotated exercise_type: "
            f"annotated={expected_specific_ids}, loaded={exd_report['exercise_id']}"
        )
        assert exd_report["is_generic_fallback"] is False
    else:
        assert exd_report["is_generic_fallback"] is True

print()
print("PASS: exercise_definition step in report")
print(f"  configured exercise_id        : {configured_exercise_id}")
print(f"  configured definitions_dir    : {configured_definitions_dir}")
print(f"  annotated exercise_type values: {annotated_exercise_ids}")
print(f"  exercise_id                   : {exd_report['exercise_id']}")
print(f"  is_generic_fallback           : {exd_report['is_generic_fallback']}")

{
  "exercise_id": "draft_squat",
  "display_name": "Draft Squat",
  "version": "draft",
  "is_generic_fallback": false,
  "laterality": "bilateral_symmetric",
  "primary_plane": "sagittal",
  "compensation_candidates": [
    "knee_valgus",
    "knee_varus",
    "asymmetric_depth",
    "excessive_trunk_flexion",
    "lateral_pelvic_shift",
    "heel_lift",
    "pelvis_rotation",
    "tempo_instability",
    "foot_external_rotation_proxy"
  ],
  "camera_protocol": {
    "recommended_zones": [
      "Z2",
      "Z8"
    ],
    "recommended_height": "H2",
    "anchor": "reference_mat",
    "distance_cm": [
      200,
      250
    ],
    "primary_observation_purpose": [
      "frontal_knee_alignment",
      "hip_flexion_depth",
      "bilateral_support_symmetry"
    ],
    "out_of_zone_policy": "warn_and_continue",
    "coordinate_correction": "none"
  },
  "filming_condition": {
    "available": true,
    "recommended_zones": [
      "Z2",
      "Z8"
    ],
    "observed_zones": [
      

## Check Summary

This notebook is a compact execution/QC checkpoint for the selected exercise definition. Negative loader fixtures and fallback behavior are covered by pytest, not this user-facing stage check.
